# Mini-project A  Flagging fraud in benefit claims

**Case:** a benefits programme wants to flag claims for investigation. The team can review **50 claims a week**.

**What your team presents (3 minutes, one point each):** what the model does, how well it works, where it fails, the value for service users, where a person reviews the output, and your recommendation. The last step of this notebook builds the summary for you.

To keep your choices, use *File, Save a copy in Drive* first.

**The data used here:**
- `benefit_claims.csv`: [view as a table](https://github.com/u1i/ml-foundations-workshop/blob/main/data/benefit_claims.csv) (opens on GitHub) · [download](https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/benefit_claims.csv) (right-click, *Save link as*)

**How this works:** each step is one cell. Click the &#9654; button on the left of the cell and read what appears below it. Where a cell has a field on the right, choose an option and click &#9654; again. No coding is needed. The cells marked *Stretch* at the end are optional and for those who like to code.

All data is synthetic.

In [ ]:
#@title Setup  (Click &#9654; first)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display, HTML
DATA = "https://raw.githubusercontent.com/u1i/ml-foundations-workshop/main/data/"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"font.size": 13, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
RED, INK, GREY, GREEN, BLUE = "#A12527", "#1d1d1b", "#6e6e6a", "#2e7d4f", "#1f5a8a"
_STYLE = {"info": ("#1f5a8a", "#eef4f9"), "see": ("#A12527", "#fbeeee"), "ok": ("#2e7d4f", "#e9f5ee"),
          "try": ("#9a6a00", "#fbf3dd"), "step": ("#1d1d1b", "#f6f5f2")}

def box(title, text="", kind="info"):
    c, bg = _STYLE[kind]
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;border-left:6px solid {c};background:{bg};'
                 f'color:#1d1d1b;padding:14px 20px;margin:12px 0;font-size:17px;line-height:1.5;max-width:900px">'
                 f'<div style="font-size:20px;font-weight:bold;color:{c};margin-bottom:4px">{title}</div>{text}</div>'))

def big(*items):
    """big((value, label), (value, label, colour), ...): large numbers in one row."""
    cells = "".join(f'<div style="display:inline-block;margin:6px 40px 6px 0;vertical-align:top"><div style="font-size:15px;color:#6e6e6a">{it[1]}</div>'
                    f'<div style="font-size:44px;font-weight:bold;color:{it[2] if len(it) > 2 else INK}">{it[0]}</div></div>' for it in items)
    display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;color:#1d1d1b">{cells}</div>'))

def answer(choice, options):
    """options: {option: (is_right, explanation)}; shows feedback for the chosen option."""
    if choice not in options:
        box("Make your choice", "Pick an option in the field on the right of this cell, then click &#9654; again.", "try")
        return False
    right, why = options[choice]
    box("Correct" if right else "Not quite", why, "ok" if right else "try")
    return right

def table(df, highlight=None, n=10):
    """Show rows with cells highlighted: highlight = {(row_label, column): colour}."""
    sub = df.head(n).astype(object).where(df.head(n).notna(), "(empty)")
    def colour(_):
        out = pd.DataFrame("", index=sub.index, columns=sub.columns)
        for (r, c), col in (highlight or {}).items():
            if r in out.index and c in out.columns:
                out.loc[r, c] = f"background-color:{col};font-weight:bold"
        return out
    display(sub.style.apply(colour, axis=None).set_table_styles([{"selector": "td,th", "props": "font-size:14px;padding:4px 10px"}]))
import ipywidgets as W
def fslider(desc, lo, hi, step, val, fmt=".2f"):
    return W.FloatSlider(value=val, min=lo, max=hi, step=step, description=desc, continuous_update=False, readout_format=fmt,
                         layout=W.Layout(width="700px"), style={"description_width": "200px"})
def islider(desc, lo, hi, val):
    return W.IntSlider(value=val, min=lo, max=hi, description=desc, continuous_update=False,
                       layout=W.Layout(width="700px"), style={"description_width": "200px"})
live = W.interact

def matrix(y_true, y_flag, labels=("honest", "fraud"), flag_word="flagged"):
    """Confusion matrix as four large boxes."""
    y_true, y_flag = np.asarray(y_true).astype(bool), np.asarray(y_flag).astype(bool)
    tp, fn = int((y_true & y_flag).sum()), int((y_true & ~y_flag).sum())
    fp, tn = int((~y_true & y_flag).sum()), int((~y_true & ~y_flag).sum())
    cell = lambda n, t, c, bg: (f'<td style="border:2px solid #1d1d1b;padding:14px 18px;background:{bg};width:260px">'
                                f'<div style="font-size:15px;color:{c};font-weight:bold">{t}</div><div style="font-size:38px;font-weight:bold;color:{c}">{n:,}</div></td>')
    display(HTML(f'<table style="font-family:Arial,Helvetica,sans-serif;border-collapse:collapse;color:#1d1d1b;margin:8px 0">'
                 f'<tr><td></td><th style="padding:6px;font-size:15px">Model: {flag_word}</th><th style="padding:6px;font-size:15px">Model: not {flag_word}</th></tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[1]}</th>{cell(tp, "Found", GREEN, "#e9f5ee")}{cell(fn, "Missed", RED, "#fbeeee")}</tr>'
                 f'<tr><th style="padding:6px;font-size:15px;text-align:right">Really {labels[0]}</th>{cell(fp, "Checked for nothing", RED, "#fbeeee")}{cell(tn, "Correctly left alone", GREEN, "#e9f5ee")}</tr></table>'))
    return tp, fn, fp, tn

def bars(values, title, highlight=None, fmt="{:.2f}", color=INK):
    """Horizontal bars with values; values is a dict or Series."""
    s = pd.Series(values)
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1))
    ax.barh(s.index.astype(str), s.values, color=[RED if k == highlight else color for k in s.index])
    for i, v in enumerate(s.values): ax.text(v, i, " " + fmt.format(v), va="center", fontsize=12)
    ax.invert_yaxis(); ax.set_title(title); plt.show()


box("Ready", "Setup done. Continue with Step 1 below.", "ok")

In [ ]:
#@title Step 1  The data
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score
df = pd.read_csv(DATA + "benefit_claims.csv"); CASE = "Flagging fraud in benefit claims"
ALL = [c for c in df.columns if c not in ("claim_id", "week", "fraud")]
big((f"{len(df):,}", "claims, 20 weeks"), (int(df["fraud"].sum()), "fraudulent", RED))
display((df.groupby("fraud")[ALL].mean(numeric_only=True).T.rename(columns={0: "honest (average)", 1: "fraud (average)"})).round(1))
box("Your first look", "Compare the two columns: where do fraudulent claims differ from honest ones? Use this to choose inputs in step 2.", "info")

In [ ]:
#@title Step 2  Build your model  (make your choices on the right, then click &#9654;)
program_type = True #@param {type:"boolean"}
claim_type = True #@param {type:"boolean"}
household_size = True #@param {type:"boolean"}
claim_amount_aed = True #@param {type:"boolean"}
claims_last_12_months = True #@param {type:"boolean"}
days_since_registration = True #@param {type:"boolean"}
payment_details_update_count = True #@param {type:"boolean"}
address_changes_12m = True #@param {type:"boolean"}
income_information_declared = True #@param {type:"boolean"}
submission_hour = True #@param {type:"boolean"}
claims_from_same_device = True #@param {type:"boolean"}
model_type = "Random forest" #@param ["Random forest", "Gradient boosting", "Decision tree (depth 4)"]
balance_classes = False #@param {type:"boolean"}
USE = [c for c in ALL if globals()[c]]
X = pd.get_dummies(df[USE], dtype=int); X = X.drop(columns=[c for c in X.columns if c.endswith("_No")]); y = df["fraud"]; tr = df["week"] <= 15; WEEKS = 5
kw = {"class_weight": "balanced"} if balance_classes and model_type != "Gradient boosting" else {}
model = {"Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42, **kw),
         "Gradient boosting": GradientBoostingClassifier(random_state=42),
         "Decision tree (depth 4)": DecisionTreeClassifier(max_depth=4, random_state=42, **kw)}[model_type].fit(X[tr], y[tr])
score = pd.Series(model.predict_proba(X[~tr])[:, 1], index=X[~tr].index)
flag = pd.Series(False, index=score.index); flag[score.sort_values(ascending=False).index[:50 * WEEKS]] = True
tp, fn, fp, tn = matrix(y[~tr], flag)
big((f"{precision_score(y[~tr], flag):.0%}", "precision at 50 a week"), (f"{recall_score(y[~tr], flag):.0%}", "recall at 50 a week", GREEN), (len(USE), "inputs used"))
box("Try changes", "Untick inputs, switch the model, or balance the classes on the right, and click &#9654; again. Note which choice finds the most fraud at 50 claims a week.", "try")

In [ ]:
#@title Step 3  Where does it fail?
test = df[~tr].assign(flagged=flag.values)
by = test.groupby("claim_type").agg(claims=("fraud", "size"), fraud=("fraud", "sum"), flagged=("flagged", "sum"),
                                     found=("fraud", lambda s: int((s & test.loc[s.index, "flagged"]).sum())))
by["share flagged"] = (by["flagged"] / by["claims"]).map("{:.1%}".format); by["fraud found"] = (by["found"] / by["fraud"].clip(lower=1)).map("{:.0%}".format)
display(by[["claims", "fraud", "flagged", "share flagged", "fraud found"]])
box("Questions for your team", "Is one type of claim flagged much more often than others? Is fraud in one type mostly missed? Would that be fair to the families affected?", "info")

In [ ]:
#@title Step 4  Explain one flagged claim  (pick one of the ten highest scores, then click &#9654;)
which = 1 #@param {type:"slider", min:1, max:10, step:1}
try:
    import shap
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "-q", "install", "shap"], check=True); import shap
idx = score.sort_values(ascending=False).index[which - 1]
ex = shap.TreeExplainer(model)(X.loc[[idx]])
ex = ex[0, :, 1] if len(ex.shape) == 3 else ex[0]
big((df.loc[idx, "claim_id"], "claim"), (f"{score[idx]:.0%}", "risk score"), ("fraud" if df.loc[idx, "fraud"] else "honest", "actually", RED if df.loc[idx, "fraud"] else GREEN))
shap.plots.waterfall(ex, max_display=8)

In [ ]:
#@title Step 5  Your team's summary  (fill in the fields on the right, then click &#9654;)
what_it_does = "" #@param {type:"string"}
how_well = "" #@param {type:"string"}
where_it_fails = "" #@param {type:"string"}
value_for_users = "" #@param {type:"string"}
where_a_person_reviews = "" #@param {type:"string"}
recommendation = "" #@param {type:"string"}
pts = [("What the model does", what_it_does), ("How well it works", how_well), ("Where it fails", where_it_fails),
       ("Value for service users", value_for_users), ("Where a person reviews the output", where_a_person_reviews), ("Our recommendation", recommendation)]
html = "".join(f'<div style="border-top:1px solid #d9d9d6;padding:10px 0"><div style="font-size:14px;color:#6e6e6a">{i + 1}  {t}</div>'
               f'<div style="font-size:19px">{v or "<i style=color:#A12527>to fill in</i>"}</div></div>' for i, (t, v) in enumerate(pts))
display(HTML(f'<div style="font-family:Arial,Helvetica,sans-serif;max-width:900px;border:2px solid #1d1d1b;padding:18px 24px;color:#1d1d1b;background:#fff">'
             f'<div style="font-size:24px;font-weight:bold;margin-bottom:6px">Our team: {CASE}</div>{html}</div>'))